# VMC 梯度计算对比分析：NetKet vs 自定义 JAX 实现

## 问题描述

在使用自定义 JAX 实现 force-based 梯度计算时，发现与 NetKet 原版的 `forces_expect_hermitian` 计算结果不一致。

## 梯度公式

变分量子蒙特卡洛中，基态能量梯度公式为：
$$
\nabla_{\theta}E = \mathbb{E}_{\pi}\left[ \left(E_{\text{loc}}-\mathbb{E}_{\pi}\left[ E_{\text{loc}} \right] \right) \left( \mathcal{O}_{\psi} + \mathcal{O}_{\psi^*} \right) \right]
$$

在 NetKet 实现中，对于 Hermitian 算符，简化为：
$$
\nabla \langle E \rangle = \langle (E_{loc} - \langle E \rangle) \nabla \log \psi^* \rangle
$$

## NetKet API 实现路径

根据源码分析，`vstate.expect_and_grad(ha)` 的调用链如下：

1. `MCState.expect_and_grad` → `expect_and_grad.dispatch`
2. `expect_and_grad_default_formula` → `expect_and_forces` + `force_to_grad`
3. `expect_and_forces` → `forces_expect_hermitian`
4. `forces_expect_hermitian` 使用 `nkjax.vjp` 计算复数梯度

关键函数位于：
- `vqs/mc/mc_state/expect_forces.py`: `forces_expect_hermitian`
- `utils/jax/_utils_vjp.py`: `vjp` (支持 R→R, R→C, C→C, C→R)

## 关键差异分析

自定义实现与 NetKet 原版的主要差异：

1. **VJP 计算方式**：NetKet 使用 `nkjax.vjp`，根据函数类型自动选择最优路径
2. **复数梯度处理**：NetKet 的 `vjp_cc` 专门处理 C→C 函数
3. **共轭处理**：NetKet 在 VJP 中使用 `conjugate=True`
4. **梯度转换**：`force_to_grad` 会对结果乘以 2（对于复数参数）

In [1]:
# ===================== 环境配置 =====================
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from jax import flatten_util

print(f"JAX version: {jax.__version__}")
print(f"NetKet version: {nk.__version__}")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.5.3
NetKet version: 3.18


In [2]:
# ===================== 1. H₂ 分子定义 & FCI 基准 =====================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能：{exc:.4f} eV")

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


In [3]:
# ===================== 2. NetKet 哈密顿量和采样器 =====================
ha = nkx.operator.from_pyscf_molecule(mol)

hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)

print(f"Hilbert space size: {hi.size}")

Hilbert space size: 4


In [4]:
# ===================== 3. 神经网络 Ansatz =====================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

In [5]:
# ===================== 4. 包装模型为 machine 函数 =====================
def create_machine(model: nnx.Module):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)
    
    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)
    
    return machine, graphdef, state

In [6]:
# ===================== 5. 原始自定义实现（有 bug） =====================
@partial(jax.jit, static_argnames=("machine",))
def compute_local_energies(machine, params, sigma):
    """
    计算局部能量 E_loc(σ) = Σ_η H(σ→η) ψ(η)/ψ(σ)
    """
    eta, H_eta = ha.get_conn_padded(sigma)
    logpsi_sigma = machine(params, sigma)
    logpsi_eta = machine(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_eta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)


def statistics(x):
    """计算样本统计量"""
    mean = jnp.mean(x)
    var = jnp.var(x)
    return mean, jnp.sqrt(var / x.shape[0])


@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian_original(machine, params, sigma):
    """
    原始实现（与 NetKet 不一致）
    
    问题：
    1. 使用 jax.grad 而非 nkjax.vjp
    2. 没有正确处理复数梯度的共轭
    3. 缺少 force_to_grad 的转换
    """
    # 1. 局域能量
    O_loc = compute_local_energies(machine, params, sigma)
    
    # 2. 能量均值
    O_mean, O_std = statistics(O_loc)
    
    # 3. 中心化
    O_centered = O_loc - O_mean
    n_samples = sigma.shape[0]
    
    def log_psi_conj(p, x):
        return jnp.conj(machine(p, x))  # log ψ*

    _, vjp_fun = jax.vjp(lambda p: log_psi_conj(p, sigma), params)
    grad = vjp_fun(O_centered / n_samples)[0]

    return O_mean, O_std, grad

In [7]:
# ===================== 6. 修复后的实现（匹配 NetKet） =====================
import netket.jax as nkjax

@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian_fixed(machine, params, sigma, model_state=None):
    """
    修复版本：匹配 NetKet 原版 forces_expect_hermitian
    
    关键修复点：
    1. 使用 nkjax.vjp 替代 jax.vjp（正确处理复数）
    2. 添加 conjugate=True 参数
    3. 使用 model_apply_fun 的标准形式
    """
    n_samples = sigma.shape[0]
    
    # 1. 计算局部能量
    O_loc = compute_local_energies(machine, params, sigma)
    
    # 2. 统计能量均值
    O_mean, O_std = statistics(O_loc)
    
    # 3. 中心化局部能量
    O_centered = O_loc - O_mean
    
    # 4. 使用 nkjax.vjp 计算梯度（关键修复）
    # 这对应 NetKet 的：
    # _, vjp_fun, *new_model_state = nkjax.vjp(
    #     lambda w: model_apply_fun({"params": w, **model_state}, σ, mutable=mutable),
    #     parameters,
    #     conjugate=True,
    #     has_aux=is_mutable,
    # )
    
    def model_apply_fun(pars, x):
        return machine(pars, x)
    
    _, vjp_fun = nkjax.vjp(
        lambda p: model_apply_fun(p, sigma),
        params,
        conjugate=True,  # 关键：共轭处理
        has_aux=False
    )
    
    # 5. 计算梯度
    # NetKet 原版：Ō_grad = vjp_fun(jnp.conjugate(O_loc) / n_samples)[0]
    Ō_grad = vjp_fun(jnp.conjugate(O_centered) / n_samples)[0]
    
    # 6. 应用 force_to_grad 转换（NetKet 在 expect_and_grad 中调用）
    # force_to_grad 会对复数参数乘以 2
    Ō_grad = jax.tree_util.tree_map(lambda x: 2 * x, Ō_grad)
    
    return O_mean, O_std, Ō_grad

In [8]:
# ===================== 7. 初始化模型 =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)
machine, graphdef, initial_params = create_machine(model)

# 创建 NetKet MCState
vstate = nk.vqs.MCState(
    sampler=sampler,
    model=model,
    n_samples=1000,
    seed=1
)

# 预热采样器
for _ in range(10):
    _ = vstate.sample()

print("模型初始化完成!")
print(f"样本形状：{vstate.samples.shape}")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/vqs/mc/mc_state/state.py:300: UserWarning: n_samples=1000 (1000 per device/MPI rank) does not divide n_chains=16, increased to 1008 (1008 per device/MPI rank)
  self.n_samples = n_samples


模型初始化完成!
样本形状：(16, 63, 4)


In [10]:
# ===================== 8. 对比梯度计算结果 =====================
# NetKet 原版
print("\n" + "="*60)
print("NetKet 原版 expect_and_grad")
print("="*60)
value_nk, grad_nk = vstate.expect_and_grad(ha)
print(f"能量：{value_nk.mean.real:.8f}")
print(f"梯度类型：{type(grad_nk)}")

# 自定义原始实现
print("\n" + "="*60)
print("自定义原始实现（有 bug）")
print("="*60)
samples_flat = vstate.samples.reshape(-1, 4)
energy_orig, energy_std_orig, grad_orig = forces_expect_hermitian_original(
    machine, initial_params, samples_flat
)
print(f"能量：{energy_orig.real:.8f} ± {energy_std_orig:.6f} Ha")
print(f"梯度类型：{type(grad_orig)}")

# 自定义修复实现
print("\n" + "="*60)
print("自定义修复实现")
print("="*60)
energy_fix, energy_std_fix, grad_fix = forces_expect_hermitian_fixed(
    machine, initial_params, samples_flat
)
print(f"能量：{energy_fix.real:.8f} ± {energy_std_fix:.6f} Ha")
print(f"梯度类型：{type(grad_fix)}")


NetKet 原版 expect_and_grad
调用了./vqs/nc_state/state.py 下的expect_and_grad 函数
调用了./vqs/mc_state/expect_grad.py 下的expect_and_grad_default_formula 函数
调用了./vqs/mc/mc_state/expect_forces.py 下的expect_and_forces 函数
调用了./vqs/mc/mc_state/expect.py中的get_local_kernel
能量：-0.48078190
梯度类型：<class 'dict'>

自定义原始实现（有 bug）
能量：-0.48078190 ± 0.007738 Ha
梯度类型：<class 'flax.nnx.statelib.State'>

自定义修复实现
能量：-0.48078190 ± 0.007738 Ha
梯度类型：<class 'flax.nnx.statelib.State'>


In [11]:
# ===================== 9. 梯度对比分析 =====================
print("\n" + "="*60)
print("梯度对比")
print("="*60)

# 展平梯度进行比较
grad_nk_flat, _ = flatten_util.ravel_pytree(grad_nk)
grad_orig_flat, _ = flatten_util.ravel_pytree(grad_orig)
grad_fix_flat, _ = flatten_util.ravel_pytree(grad_fix)

print(f"\nNetKet 梯度范数：{jnp.linalg.norm(grad_nk_flat):.8f}")
print(f"原始实现梯度范数：{jnp.linalg.norm(grad_orig_flat):.8f}")
print(f"修复实现梯度范数：{jnp.linalg.norm(grad_fix_flat):.8f}")

print(f"\nNetKet vs 原始差异：{jnp.linalg.norm(grad_nk_flat - grad_orig_flat):.8f}")
print(f"NetKet vs 修复差异：{jnp.linalg.norm(grad_nk_flat - grad_fix_flat):.8f}")

# 检查是否匹配
tolerance = 1e-6
if jnp.allclose(grad_nk_flat, grad_fix_flat, rtol=tolerance, atol=tolerance):
    print(f"\n✅ 修复实现与 NetKet 匹配！(tol={tolerance})")
else:
    print(f"\n❌ 修复实现与 NetKet 仍有差异")
    print(f"最大绝对差异：{jnp.max(jnp.abs(grad_nk_flat - grad_fix_flat)):.8e}")


梯度对比

NetKet 梯度范数：0.92625589
原始实现梯度范数：0.46312795
修复实现梯度范数：0.92625589

NetKet vs 原始差异：0.93410798
NetKet vs 修复差异：0.00000000

✅ 修复实现与 NetKet 匹配！(tol=1e-06)


In [12]:
# ===================== 10. 详细分析差异来源 =====================
print("\n" + "="*60)
print("差异来源分析")
print("="*60)

# 1. 检查 VJP 计算的差异
print("\n1. VJP 计算方式对比:")
print("   - 原始实现：jax.vjp + jnp.conj(machine(...))")
print("   - NetKet: nkjax.vjp(conjugate=True)")
print("   - nkjax.vjp 会根据函数类型自动选择最优路径 (R→R, R→C, C→C, C→R)")

# 2. 检查 force_to_grad 的影响
print("\n2. force_to_grad 的影响:")
print("   - force_to_grad 会对结果乘以 2")
print("   - 对于复数参数：grad = 2 * F")
print("   - 对于实数参数：grad = 2 * Re[F]")

# 3. 数值验证
print("\n3. 数值验证:")
ratio_nk_orig = jnp.linalg.norm(grad_nk_flat) / jnp.linalg.norm(grad_orig_flat)
ratio_nk_fix = jnp.linalg.norm(grad_nk_flat) / jnp.linalg.norm(grad_fix_flat)
print(f"   NetKet/原始梯度范数比：{ratio_nk_orig:.4f}")
print(f"   NetKet/修复梯度范数比：{ratio_nk_fix:.4f}")

if jnp.isclose(ratio_nk_orig, 2.0, rtol=0.1):
    print("   → 原始实现缺少 force_to_grad 的 2 倍因子!")


差异来源分析

1. VJP 计算方式对比:
   - 原始实现：jax.vjp + jnp.conj(machine(...))
   - NetKet: nkjax.vjp(conjugate=True)
   - nkjax.vjp 会根据函数类型自动选择最优路径 (R→R, R→C, C→C, C→R)

2. force_to_grad 的影响:
   - force_to_grad 会对结果乘以 2
   - 对于复数参数：grad = 2 * F
   - 对于实数参数：grad = 2 * Re[F]

3. 数值验证:
   NetKet/原始梯度范数比：2.0000
   NetKet/修复梯度范数比：1.0000
   → 原始实现缺少 force_to_grad 的 2 倍因子!


## 结论

### 梯度不一致的原因

1. **VJP 计算方式不同**：
   - 原始实现使用 `jax.vjp` 直接计算
   - NetKet 使用 `nkjax.vjp`，根据函数类型自动选择最优路径
   - 对于 C→C 函数，`nkjax.vjp_cc` 有更高效的实现

2. **共轭处理**：
   - 原始实现手动对输出取共轭：`jnp.conj(machine(...))`
   - NetKet 使用 `conjugate=True` 参数，在 VJP 内部处理

3. **force_to_grad 转换**：
   - `expect_and_grad` 在 `expect_and_forces` 后会调用 `force_to_grad`
   - `force_to_grad` 对梯度乘以 2
   - 这是为了匹配变分原理中的因子 2

### 修复方案

要完全匹配 NetKet 的行为，需要：

1. 使用 `nkjax.vjp` 替代 `jax.vjp`
2. 设置 `conjugate=True`
3. 对结果应用 `force_to_grad` 转换（乘以 2）

### NetKet 完整调用链

```
vstate.expect_and_grad(ha)
  ↓
expect_and_grad.dispatch (vqs/mc/mc_state/expect_grad.py)
  ↓
expect_and_forces (vqs/mc/mc_state/expect_forces.py)
  ↓
forces_expect_hermitian
  - 计算 O_loc
  - 计算 O_centered = O_loc - ⟨O⟩
  - 使用 nkjax.vjp(conjugate=True) 计算 VJP
  - Ō_grad = vjp_fun(conj(O_centered) / N)
  ↓
force_to_grad
  - grad = 2 * Ō_grad
```

### 参考文档

- [VMC 的梯度计算.md](../0504/VMC 的梯度计算.md) - 梯度公式推导
- [目前的 API 调查进展.md](../0424/目前的 API 调查进展.md) - NetKet 源码分析
- [VMC_jax_native.ipynb](../0504/VMC_jax_native.ipynb) - 纯 JAX 实现参考